# seq2seq 模式介绍



<img src="./markdown-note/images/img_18.png" width="500"/>




图中表示的是一个中文到英文的翻译：欢迎来北京→welcome to BeiJing。编码器首先处理中文输入"欢迎来北京”，通过GRU模型获得每个时间步的输出张量，最后将它们拼接成一个中间语义张量c；接着解码器将使用这个中间语义张量c以及每一个时间步的隐层张量，逐个生成对应的翻译语言

早期在解决机器翻译这一类seq2seq问题时，通常采用的做法是利用一个编码器（Encoder）和一个解码器（Decoder）构建端到端的神经网络模型，但是基于编码解码的神经网络存在两个问题：

问题1：如果翻译的句子很长很复，比如直接一篇文章输进去，模型的计算量很大，并且模型的准确率下降严重。

问题2：在翻译时，可能在不同的语境下，同一个词具有不同的含义，但是网络对这些词向量并没有区分度，没有考虑词与词之间的相关性，导致翻译效果比较差。

主要就是每次都用这个中间语义张量 C 来生成翻译语言 ，但是这个中间语义张量 C 并没有考虑词与词之间的相关性，也没有侧重点

比如机器翻译任务，输入source为：Tom chase Jerry，输出target为：“汤姆”，“追逐”，“杰瑞”。在翻译"Jerry"这个词为中文的时候，普通Encoder-Decoder框架中，source里的每个单词对翻译目标单词"杰瑞"贡献是相同的，很明显这里不太合理，显然"Jerry"对于翻译成"杰瑞"更重要。

# 注意力机制

通俗来讲就是对于模型的每一个输入顶，可能是图片中的不同部分，或者是语句中的某个单词分配一个权重，这个权重的大小就代表了我们希望模型对该部分一个关注程度。这样一来，通过权重大小来模拟人在处理信息的注意力的侧重，有效的提高了模型的性能，并且一定程度上降低了计算量。

深度学习中的注意力机制通常可分为三类：软注意（全局注意）、硬注意（局部注意）和自注意 (内注意）

- 软注意机制(Soft/GlobalAttention:对每个输入项的分配的权重为0-1之间，也就是某些部分关注的多一点，某些部分关注的少一点，因为对大部分信息都有考虑，但考虑程度不一样，所以
相对来说计算量比较大。

- 硬注意机制（Hard/LocalAttention,[了解即可])：对每个输入项分配的权重非o即1，和软注意不同，硬注意机制只考虑那部分需要关注，哪部分不关注，也就是直接舍弃掉一些不相关项。优
势在于可以减少一定的时间和计算成本，但有可能丢失掉一些本应该注意的信息。

- 自注意机制（Self/IntraAttention)：对每个输入项分配的权重取决于输入项之间的相互作用，即通过输入项内部的"表决"来决定应该关注哪些输入项。和前两种相比，在处理很长的输入
时，具有并行计算的优势。

## encoder-decoder 模式加入注意力机制

普通的Encoder和Decoder 框架如下图所示：

<img src="./markdown-note/images/img_19.png" width="500"/>">

上图图例可以把它看作由一个句子（或篇章）生成另外一个句子（或篇章）的通用处理模型。
对于句子对，我们的目标是给定输入句子Source，期待通过Encoder-Decoder框架来生成目标句子Target。
Source和Target可以是同一种语言，也可以是两种不同的语言。而Source和Target分别由各自的单词序列构成：

Source =(X1,X2...Xm)

Target = (yn, y2...yn)

那么中间语义张量C可以表示为：

C = F(X1,X2...Xm)


对于解码器Decoder来说，其任务是根据句子Source的中间语义表示C和之前已经生成的历史信息,y1, y_2..y_i-1来生成i时刻要生成的单词y_i


y_i = G(C,y_1, y_2..y_i-1)

由此可得

y_0 = f(C)

y_1 = f(C,y_0)

y_2 = f(C,y_0,y_1)

其中f 是 Decoder的非线性变换函数。从这里可以看出，在生成目标句子的单词时，不论生成哪个单词，它们使用的输入句子Source的语义编码C都是一样的，没有任何区别。而语义编码C又是通过对source经过Encoder编码产生的，因此对于target中的任何一个单词，source中任意单词对某个目标单词y_i来说影响力都是相同的，这就是为什么说上面的模型没有体现注意力的原因。

接下来我们给这个模型加入注意力机制

举例说明，如何添加Attention:
比如机器翻译任务，输入source为：Tom chase Jerry， 输出target为：“汤姆”，“追逐”，“杰瑞”。在翻译”Jerry”为 "杰瑞" 的时候，普通Encoder-Decoder框架中，source里的每个单词对翻译目标单词“杰瑞“贡献是相同的，很明显这里不太合理，显然”Jerry”对于翻译成"杰瑞”更重要。

如果引l入Attention模型，在生成”杰瑞"的时候，应该体现出英文单词对于翻译当前中文单词不同的影响程度，比如给出类似下面一个概率分布值：（Tom,0.3）（Chase,0.2)（Jerry,0.5).每个英文单词的概率代表了翻译当前单词“杰瑞”时，注意力分配模型分配给不同英文单词的注意力大小。

因此，基于上述例子所示，对于target中任意一个单词都应该有对应的source中的单词的注意力分配概率

而且，由于注意力模型的加入，原来在生成target单词时候的中间语义C 就不再是固定的，而是会根据注意力概率变化的C，加入了注意力模型的Encoder-Decoder框架就变成了

<img src="./markdown-note/images/img_20.png" width="500"/>">


即生成目标句子单词的过程成了下面的形式：

y1= f1(C1)

y2 = f1(C2,y1)

y3 = f1(C3,y1,y2)

而每个Ci可能对应着不同的源语句子单词的注意分配概率分布，比如对于上面的英汉翻译来说，其对应的信息可能如下：

C_Tom = g(0.6 * f2(Tom), 0.2 * f2(Chase), 0.2 * f2(Jerry))

C_chase = g(0.2 * f2(Tom), 0.7 * f2(Chase), 0.1 * f2(Jerry))

C_Jerry = g(0.3 * f2(Tom), 0.2 * f2(Chase), 0.5 * f2(Jerry))

f2函数代表Encoder对输入英文单词的某种变换函数，比如如果Encoder是用的RNN模型的话，这个f2函数的结果往往是某个时刻输入后隐层节点的状态值；

g代表Encoder根据单词的中间表示合成整个句子中间语义表示的变换函数，一般的做法中，g函数就是对构成元素加权求和，即下列公式


<img src="./markdown-note/images/img_22.png" width="500"/>"

<img src="./markdown-note/images/img_23.png" width="500"/>">


## 注意力的概率计算过程

其实Attention机制可以看作，Target中每个单词是对Source每个单词的加权求和，而权重是Source中每个单词对Target中每个单词的重要程度。因此，Attention的本质思想会表示成下图：

<img src="./markdown-note/images/img_30.png" width="1200"/>"

我们把上图的过程 改为公式表示如下：

<img src="./markdown-note/images/img_28.png" width="500"/>


在上面的这个翻译的例子里面我们KEY 和 VALUE 都是h_i  但是需要知道的是 不一定在所有场景中这两者都是相同的 **也就是说参与相似度计算最终得到权重的 Key 和 参与加权计算注意力的 Value 可以是不同的**


在更高级的模型（比如 Transformer）里，它们是可以分开的，这是为了让模型更灵活、更强大。

我们可以把这个过程理解为：

Key 负责 “对齐”：它的任务是和 Query 计算相似度，决定 “我应该关注哪里”。它更像是一个 “注意力探测器”。

Value 负责 “提供信息”：它的任务是被加权求和，提供 “我关注到的内容”。它更像是一个 “信息存储器”。

当我们把 Key 和 Value 分开时，模型就可以学到：

用不同的方式来表示 “我是谁”（Key）和 “我有什么”（Value）。

这让模型的表达能力大大增强，可以处理更复杂的任务。



<img src="./markdown-note/images/img_29.png" width="500"/>